# Think@n Evaluation (Table 2 + Figure 5)

Reproduces **Table 2** and **Figure 5** from the DTR paper.

This notebook evaluates different inference-time strategies for allocating compute:
- **Majority@n**: Generate n responses, take majority vote
- **Best-of-n**: Generate n responses, select the one with highest metric score
- **Think@n**: DTR-guided strategy that selectively extends thinking for uncertain samples
- **Cons@n**: Consensus-based selection using DTR

The key finding is that Think@n achieves higher accuracy at lower cost than baselines.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style="whitegrid")

from dtr.aggregation.strategies import (
    SampleResult,
    run_all_strategies,
    run_trials,
    think_at_n,
    cons_at_n,
)
from dtr.aggregation.cost import compute_cons_cost, compute_selective_cost, compute_cost_ratio
from dtr.utils.io import load_json

## Configuration

In [ ]:
MODELS = ["deepseek-r1", "qwq-32b"]
BENCHMARKS = ["aime_2025", "hmmt_2025", "gpqa_diamond", "livecodebench"]
BENCHMARK_LABELS = ["AIME 2025", "HMMT 2025", "GPQA Diamond", "LiveCodeBench"]

METRICS_DIR = Path("..") / "results" / "metrics"
N_VALUES = [1, 4, 8, 16, 32]
N_TRIALS = 100  # Number of bootstrap trials for confidence intervals

STRATEGIES = [
    "majority@n",
    "best_of_n_token_count",
    "best_of_n_dtr",
    "think@n",
    "cons@n",
    "selective@n",
]

## Load Per-Sample Results

Load generation results including DTR scores, token counts, correctness, and answers.

In [ ]:
all_results = {}

for model in MODELS:
    for bench in BENCHMARKS:
        key = f"{model}/{bench}"
        metrics_path = METRICS_DIR / model / bench / "per_sample_metrics.json"
        
        if metrics_path.exists():
            data = load_json(str(metrics_path))
            all_results[key] = data
            print(f"Loaded {key}: {len(data)} samples")
        else:
            print(f"WARNING: {metrics_path} not found")

print(f"\nTotal combinations loaded: {len(all_results)}")

## Run All Strategies

For each model-benchmark pair, run all aggregation strategies with bootstrap trials.

In [ ]:
strategy_results = {}

for key, data in all_results.items():
    print(f"Processing {key}...")
    
    # Convert to SampleResult objects
    samples = [
        SampleResult(
            answer=d.get("answer", ""),
            correct=d["correct"],
            dtr=d.get("dtr", 0.0),
            token_count=d.get("token_count", 0),
            log_prob=d.get("log_prob", 0.0),
        )
        for d in data
    ]
    
    # Run strategies with trials for confidence intervals
    results = run_all_strategies(
        samples,
        n_values=N_VALUES,
        n_trials=N_TRIALS,
        strategies=STRATEGIES,
    )
    strategy_results[key] = results

print("\nAll strategies computed.")

## Table 2: Accuracy and Cost by Strategy

Shows accuracy and normalized inference cost for each strategy across benchmarks.
Think@n should achieve competitive accuracy with lower cost.

In [ ]:
DISPLAY_MODEL = MODELS[0]
DISPLAY_N = 16  # n value for the table

table_rows = []
for strategy in STRATEGIES:
    row = {"Strategy": strategy}
    for bench, bench_label in zip(BENCHMARKS, BENCHMARK_LABELS):
        key = f"{DISPLAY_MODEL}/{bench}"
        if key not in strategy_results:
            row[f"{bench_label} Acc"] = "-"
            row[f"{bench_label} Cost"] = "-"
            continue
        
        res = strategy_results[key]
        if strategy in res and DISPLAY_N in res[strategy]:
            entry = res[strategy][DISPLAY_N]
            acc = entry.get("accuracy_mean", entry.get("accuracy", 0))
            acc_std = entry.get("accuracy_std", 0)
            cost = entry.get("cost_mean", entry.get("cost", 0))
            row[f"{bench_label} Acc"] = f"{acc:.1%} +/- {acc_std:.1%}"
            row[f"{bench_label} Cost"] = f"{cost:.2f}x"
        else:
            row[f"{bench_label} Acc"] = "-"
            row[f"{bench_label} Cost"] = "-"
    
    table_rows.append(row)

table2_df = pd.DataFrame(table_rows)
table2_df.set_index("Strategy", inplace=True)

print(f"Table 2: Strategy Performance at n={DISPLAY_N} ({DISPLAY_MODEL})")
print("=" * 100)
table2_df

## Figure 5: Pareto Frontier (Accuracy vs Cost)

Each strategy is plotted as a curve across different n values. Points closer to the
upper-left corner are better (higher accuracy, lower cost). Think@n should dominate
the Pareto frontier.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
fig.suptitle(
    f"Figure 5: Accuracy vs Inference Cost ({DISPLAY_MODEL})",
    fontsize=15, fontweight="bold", y=1.05,
)

strategy_colors = {
    "majority@n": sns.color_palette()[0],
    "best_of_n_token_count": sns.color_palette()[1],
    "best_of_n_dtr": sns.color_palette()[2],
    "think@n": sns.color_palette()[3],
    "cons@n": sns.color_palette()[4],
    "selective@n": sns.color_palette()[5],
}

strategy_markers = {
    "majority@n": "o",
    "best_of_n_token_count": "s",
    "best_of_n_dtr": "D",
    "think@n": "*",
    "cons@n": "^",
    "selective@n": "v",
}

strategy_labels = {
    "majority@n": "Majority@n",
    "best_of_n_token_count": "BoN (Token Count)",
    "best_of_n_dtr": "BoN (DTR)",
    "think@n": "Think@n",
    "cons@n": "Cons@n",
    "selective@n": "Selective@n",
}

for ax, bench, bench_label in zip(axes, BENCHMARKS, BENCHMARK_LABELS):
    key = f"{DISPLAY_MODEL}/{bench}"
    if key not in strategy_results:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(bench_label, fontsize=12)
        continue
    
    res = strategy_results[key]
    
    for strategy in STRATEGIES:
        if strategy not in res:
            continue
        
        costs = []
        accs = []
        acc_errs = []
        
        for n in sorted(res[strategy].keys()):
            entry = res[strategy][n]
            costs.append(entry.get("cost_mean", entry.get("cost", n)))
            accs.append(entry.get("accuracy_mean", entry.get("accuracy", 0)))
            acc_errs.append(entry.get("accuracy_std", 0))
        
        ax.errorbar(
            costs, accs, yerr=acc_errs,
            marker=strategy_markers.get(strategy, "o"),
            color=strategy_colors.get(strategy, "gray"),
            linewidth=1.5,
            markersize=8 if strategy == "think@n" else 6,
            capsize=2,
            label=strategy_labels.get(strategy, strategy),
            zorder=10 if strategy == "think@n" else 5,
        )
    
    ax.set_title(bench_label, fontsize=12)
    ax.set_xlabel("Normalized Inference Cost", fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("Accuracy", fontsize=11)

# Shared legend
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center",
    ncol=len(STRATEGIES),
    fontsize=9,
    bbox_to_anchor=(0.5, -0.12),
)

plt.tight_layout()
plt.savefig("../results/figures/fig5_pareto_frontier.pdf", bbox_inches="tight", dpi=300)
plt.show()

## Accuracy Scaling with n

How does accuracy scale as we increase the number of samples n for each strategy?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharey=True)
fig.suptitle(
    f"Accuracy Scaling with n ({DISPLAY_MODEL})",
    fontsize=15, fontweight="bold", y=1.05,
)

for ax, bench, bench_label in zip(axes, BENCHMARKS, BENCHMARK_LABELS):
    key = f"{DISPLAY_MODEL}/{bench}"
    if key not in strategy_results:
        ax.set_title(bench_label)
        continue
    
    res = strategy_results[key]
    
    for strategy in STRATEGIES:
        if strategy not in res:
            continue
        
        n_vals = sorted(res[strategy].keys())
        accs = [res[strategy][n].get("accuracy_mean", res[strategy][n].get("accuracy", 0)) for n in n_vals]
        
        ax.plot(
            n_vals, accs,
            marker=strategy_markers.get(strategy, "o"),
            color=strategy_colors.get(strategy, "gray"),
            linewidth=1.5,
            markersize=6,
            label=strategy_labels.get(strategy, strategy),
        )
    
    ax.set_title(bench_label, fontsize=12)
    ax.set_xlabel("n (number of samples)", fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("Accuracy", fontsize=11)
    ax.set_xscale("log", base=2)
    ax.set_xticks(N_VALUES)
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc="lower center", ncol=len(STRATEGIES),
    fontsize=9, bbox_to_anchor=(0.5, -0.12),
)

plt.tight_layout()
plt.show()